In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from scipy.stats import norm
from statsmodels.discrete.conditional_models import ConditionalPoisson


# Remark: Comments translated to English using AI.

# Years for which we process accident data.
# Accident data is available for all years; traffic data for 2018–2023 comes from the INWEVA historical dataset,
# while traffic data for 2024 and 2025 comes from separate CSV files (see the cell 'traffic per road section per year').

JAREN = range(2018, 2026)

# Structure of this notebook:
#   1. maak_baanvakken()                -> once: retrieve the motorway road sections from INWEVA (2023 network)
#   2. verwerk_jaar(jaar, baanvakken)   -> for each year: load accident data, filter it, and link it to a road section
#   3. Combine all years into a single table.

In [14]:
# 1. Create road sections (once)
# INWEVA uses the 2023 road network. Therefore, we always use wegvakken.txt from 2023,
# even when linking accidents from other years later on.


def maak_baanvakken():
    # INWEVA history: one row per road section (a stretch of roadway in one direction),
    # with daily traffic volumes for 2018–2023.
    jaarkolommen = [f"etmaal_AL_weekdag_{j}" for j in range(2018, 2024)]
    inweva = pd.read_excel("data/INWEVA/INWEVA_2023/A. Outputbestanden Excel/INWEVA_2023_historie.xlsx",
                           usecols=["VIAS_baannr", "Baantype_van", "Wegnr_van", "Ri_van", "Hm_van", "Hm_naar",
                                    "Lengte_baan_meter", "NwbId", "Traject_van", "Traject_naar"] + jaarkolommen)

    inweva_hr = inweva[inweva["Baantype_van"] == "HR"].copy()  # Main carriageways only (excluding entry and exit ramps)

    # 2023 road network: which road sections are part of a motorway's main carriageway?
    wegvakken = pd.read_csv("data/BRON/01-01-2023_31-12-2023/Netwerkgegevens/wegvakken.txt",
                            encoding="latin-1", dtype=str, usecols=["WVK_ID", "WEGBEHSRT", "BST_CODE", "ROUTELTR"])
    snelweg = wegvakken[
        (wegvakken["WEGBEHSRT"] == "R")      # Rijkswaterstaat
        & (wegvakken["BST_CODE"] == "HR")    # Main carriageway
        & (wegvakken["ROUTELTR"] == "A")     # Motorway
    ]

    # Filter for motorways only
    # INWEVA also includes N-roads, but has no "A or N" column.
    # Each road section contains a list of NWB road sections (NwbId).
    # If any of these appears in 'snelweg', the road section is classified as a motorway.
    snelweg_ids = set(snelweg["WVK_ID"])   # set: Python can look up values in this very quickly

    def is_snelweg(nwb_ids):
        # "600461021H,508588042H" -> ["600461021", "508588042"]  (create a list, remove spaces and H/T suffixes)
        ids = [x.strip().rstrip("HT") for x in nwb_ids.split(",")]
        return any(i in snelweg_ids for i in ids)   # True if at least one road section is a motorway

    baanvakken = inweva_hr[inweva_hr["NwbId"].apply(is_snelweg)].copy()

    # Put the start and end points of each road section in ascending order
    # On the left carriageway (L), kilometre markers decrease (Hm_van > Hm_naar). between() only works with (low, high).
    baanvakken["hm_laag"] = baanvakken[["Hm_van", "Hm_naar"]].min(axis=1)
    baanvakken["hm_hoog"] = baanvakken[["Hm_van", "Hm_naar"]].max(axis=1)

    # Exclude road sections spanning a kilometre jump
    # At a jump (e.g. where the A10 ring road starts again at km 0), the range is much larger than the actual road section,
    # which would cause accidents from elsewhere to be incorrectly assigned to it.
    sprong = (baanvakken["hm_hoog"] - baanvakken["hm_laag"]) > baanvakken["Lengte_baan_meter"] + 1000
    baanvakken = baanvakken[~sprong].copy()   # ~ = not: keep everything that is not a jump

    print(f"{len(baanvakken)} motorway road sections ({sprong.sum()} spanning a kilometre jump excluded)")
    return baanvakken

In [ ]:
baanvakken = maak_baanvakken()   # expected: 3209 motorway road sections (46 excluded)
baanvakken

In [ ]:
# 2. Process accident data for one year
# All data uses files from the same year (accidents, road sections, kilometre markers),
# so the road section IDs always match.

def verwerk_jaar(jaar, baanvakken):
    map_bron = f"data/BRON/01-01-{jaar}_31-12-{jaar}"

    # Read data (only the columns we need)
    # accidents: number, road section, severity (AP3_CODE), lighting (LGD_ID), speed limit (MAXSNELHD), accident type (AOL_ID)
    ongevallen = pd.read_csv(f"{map_bron}/Ongevallengegevens/ongevallen.txt", encoding="latin-1", dtype=str,
                             usecols=["VKL_NUMMER", "WVK_ID", "AP3_CODE", "LGD_ID", "MAXSNELHD", "AOL_ID"])
    # road sections: ID, managing authority, road type, route letter (A/N), road number, direction (R/L)
    wegvakken = pd.read_csv(f"{map_bron}/Netwerkgegevens/wegvakken.txt", encoding="latin-1", dtype=str,
                            usecols=["WVK_ID", "WEGBEHSRT", "BST_CODE", "ROUTELTR", "WEGNUMMER", "RPE_CODE"])

    # Select relevant rows: main carriageways of motorways only
    snelweg = wegvakken[
        (wegvakken["WEGBEHSRT"] == "R")      # Rijkswaterstaat
        & (wegvakken["BST_CODE"] == "HR")    # Main carriageway
        & (wegvakken["ROUTELTR"] == "A")     # Motorway
    ]
    snelweg = snelweg.drop(columns=["WEGBEHSRT", "BST_CODE", "ROUTELTR"])  # These are now consistent across all rows

    # Accidents on motorways
    # Merging on WVK_ID adds the road and direction to each accident; how="inner" keeps only motorway accidents
    ongevallen_snelweg = ongevallen.merge(snelweg, on="WVK_ID", how="inner")

    # Add two columns in the same format as inweva
    ongevallen_snelweg["weg"] = pd.to_numeric(ongevallen_snelweg["WEGNUMMER"])   # "004" -> 4

    # Accident location = midpoint of its road section
    # The HECTOMETER column is not always populated (2018: 36% missing, 2023: 4% missing), but the road section (WVK_ID) is.
    # An INWEVA road section consists of entire road sections, so the accident is still assigned to the correct road section
    # (tested: 99.3–99.9% assigned to the same road section as when using the actual hectometre marker). This works the same way for every year.
    hectopunten = pd.read_csv(f"{map_bron}/Netwerkgegevens/hectopunten.txt", encoding="latin-1", dtype=str,
                              usecols=["WVK_ID", "HECTOMETER"])
    hectopunten["meter"] = pd.to_numeric(hectopunten["HECTOMETER"]) * 100   # 156 -> 15600
    midden_wegvak = hectopunten.groupby("WVK_ID")["meter"].median()        # Middle marker per road section
    ongevallen_snelweg["positie_m"] = ongevallen_snelweg["WVK_ID"].map(midden_wegvak)  # Look up the position for each accident

    # Link accidents to road sections
    # Rule: an accident belongs to the road section on the same road and in the same direction
    # whose range (hm_laag to hm_hoog) contains the accident's position.
    # 1. Merge on road + direction: match each accident with ALL road sections on its road and in its direction
    kandidaten = ongevallen_snelweg.merge(
        baanvakken[["VIAS_baannr", "Wegnr_van", "Ri_van", "hm_laag", "hm_hoog"]],
        left_on=["weg", "RPE_CODE"], right_on=["Wegnr_van", "Ri_van"])
    # 2. Keep only the row where the position falls within the road section
    raak = kandidaten[kandidaten["positie_m"].between(kandidaten["hm_laag"], kandidaten["hm_hoog"])]
    # 3. If an accident lies exactly on the boundary between two road sections, keep only one
    gekoppeld = raak.drop_duplicates("VKL_NUMMER").copy()

    gekoppeld["jaar"] = jaar   # So that we know which year each accident comes from after combining the data

    print(f"{jaar}: {len(ongevallen_snelweg)} motorway accidents, {len(gekoppeld)} linked "
          f"({len(gekoppeld) / len(ongevallen_snelweg):.1%})")
    return gekoppeld

In [ ]:
# 3. Process all years and combine them into one table
# [verwerk_jaar(j, baanvakken) for j in JAREN] creates a list containing one table per year;
# pd.concat stacks them into a single table. The 'jaar' column indicates which year each accident occurred.
alle_ongevallen = pd.concat([verwerk_jaar(j, baanvakken) for j in JAREN], ignore_index=True)

# Result: one row per linked motorway accident, with the road section (VIAS_baannr) where it occurred
alle_ongevallen

In [ ]:
# Check: which road section does an accident belong to?
# Accident 20230062249 -> road section 11553 (A2 right, km 222.15–224.19, between the A73 and Echt)
voorbeeld = alle_ongevallen[alle_ongevallen["VKL_NUMMER"] == "20230062249"]
voorbeeld[["VKL_NUMMER", "jaar", "WEGNUMMER", "RPE_CODE", "positie_m", "VIAS_baannr", "hm_laag", "hm_hoog"]]

In [ ]:
# 4. Add collumn treated as a Boolean. True when speed limit changed from 120 or 130 km/h to 100 km/h, False if it remained 100 km/h, None no max speed was recorded or the speed limit was different before / after 2020. 

# AANGEPAST (Harmen): de kolom 'limiet' werd nergens in het notebook gemaakt (alleen in een losse cel die
# niet meer bestaat), waardoor deze cel bij 'Run All' een KeyError gaf. MAXSNELHD is tekst ("100") -> getal.
alle_ongevallen["limiet"] = pd.to_numeric(alle_ongevallen["MAXSNELHD"], errors="coerce")

# Median speed limit before 2020 (2018–2019)
mediaan_voor = (
    alle_ongevallen[alle_ongevallen["jaar"].isin([2018, 2019])]
    .groupby("VIAS_baannr")["limiet"]
    .median()
)

# Median speed limit after 2020 (2021–2023)
mediaan_na = (
    alle_ongevallen[alle_ongevallen["jaar"].isin([2021, 2022, 2023])]
    .groupby("VIAS_baannr")["limiet"]
    .median()
)

# Combine the medians
snelheid_df = pd.DataFrame({
    "limiet_voor_2020": mediaan_voor,
    "limiet_na_2020": mediaan_na
})

# Classify road sections
conditions = [
    snelheid_df["limiet_voor_2020"].isin([120, 130]) &
    (snelheid_df["limiet_na_2020"] == 100),

    (snelheid_df["limiet_voor_2020"] == 100) &
    (snelheid_df["limiet_na_2020"] == 100)
]

snelheid_df["treated"] = np.select(
    conditions,
    [True, False],
    default=None
)

print("True:", (snelheid_df["treated"] == True).sum())
print("False:", (snelheid_df["treated"] == False).sum())
print("NaN:", snelheid_df["treated"].isna().sum())

alle_ongevallen["treated"] = alle_ongevallen["VIAS_baannr"].map(
    snelheid_df["treated"]
)

alle_ongevallen


In [ ]:
# Number of accidents per road section per year
# Grouping by both road section and year, size() counts the number of rows (= accidents) for each combination.
# unstack places the years side by side, allowing you to compare years for each road section.
per_baanvak_jaar = alle_ongevallen.groupby(["VIAS_baannr", "jaar"]).size().unstack(fill_value=0)

# Example: road section 3901 (A4 right near Hoofddorp) -> 2018: 27, 2019: 19, 2020: 6, ... 2023: 18
per_baanvak_jaar

In [ ]:
# Traffic per road section per year

# 2018–2023: INWEVA history. Traffic volumes are stored in six separate columns (one per year).
# melt stacks them into rows: six rows per road section, with a 'jaar' column and an 'intensiteit' column.
jaarkolommen = [f"etmaal_AL_weekdag_{j}" for j in range(2018, 2024)]
verkeer = baanvakken.melt(id_vars=["VIAS_baannr", "Lengte_baan_meter"], value_vars=jaarkolommen,
                          var_name="kolom", value_name="intensiteit")
verkeer["jaar"] = verkeer["kolom"].str[-4:].astype(int)   # "etmaal_AL_weekdag_2018" -> 2018
verkeer = verkeer.drop(columns="kolom")

# UPDATED (Harmen): traffic data for 2024 and 2025 added so that all charts cover 2018–2025.
# These years use a new format (CSV files with ';' separators and a different network: 'verkeersbanen').
# Each traffic section specifies in 'vias_baan_ids' which of our road sections (VIAS_baannr) it covers.
# If multiple traffic sections cover one road section, we calculate the length-weighted average.
# This provides values for 2024 and 2025 for approximately 98% of our road sections.
def verkeer_nieuw_formaat(jaar):
    map_inweva = Path(f"data/INWEVA/INWEVA_{jaar}")

    def lees(begin):   # File names differ in capitalization between 2024 and 2025
        pad = next(p for p in map_inweva.glob("*.csv") if p.name.lower().startswith(begin))
        df = pd.read_csv(pad, sep=";", dtype=str)
        df.columns = df.columns.str.lower()
        return df

    banen = lees(f"inweva_verkeersbanen_{jaar}")
    weekdag = lees(f"inweva_weekdag_{jaar}")
    vias_kolom = next(k for k in banen.columns if k.startswith("vias"))   # 2024: vias_baan_ids, 2025: vias_baan_id

    df = banen[["vbn_id", "vbn_lengte", vias_kolom]].merge(weekdag[["vbn_id", "al_e_wk"]], on="vbn_id")
    df["VIAS_baannr"] = df[vias_kolom].str.split(",")      # "3901, 3978, 4024" -> list
    df = df.explode("VIAS_baannr")                         # One row per old road section
    df["VIAS_baannr"] = pd.to_numeric(df["VIAS_baannr"].str.strip(), errors="coerce")
    df["intensiteit"] = pd.to_numeric(df["al_e_wk"], errors="coerce")   # Average number of vehicles per day
    df["lengte"] = pd.to_numeric(df["vbn_lengte"], errors="coerce")
    df = df.dropna(subset=["VIAS_baannr", "intensiteit", "lengte"])

    df["int_x_lengte"] = df["intensiteit"] * df["lengte"]
    som = df.groupby("VIAS_baannr")[["int_x_lengte", "lengte"]].sum()
    uit = (som["int_x_lengte"] / som["lengte"]).rename("intensiteit").reset_index()
    uit["VIAS_baannr"] = uit["VIAS_baannr"].astype(int)
    uit["jaar"] = jaar
    return uit

nieuw = pd.concat([verkeer_nieuw_formaat(j) for j in (2024, 2025)], ignore_index=True)
nieuw = nieuw.merge(baanvakken[["VIAS_baannr", "Lengte_baan_meter"]], on="VIAS_baannr")   # Only our road sections
verkeer = pd.concat([verkeer, nieuw], ignore_index=True)

# UPDATED (Harmen): some road sections have an intensity of -1 (= no value). Setting these to 0 resulted in
# vehicle-kilometres = 0 even though their accidents were still counted. We now exclude these road section-years entirely.
verkeer = verkeer[verkeer["intensiteit"] >= 0]
print("Road sections with traffic data per year:", verkeer.groupby("jaar")["VIAS_baannr"].nunique().to_dict())

# Accidents per road section per year (long format, so it matches 'verkeer')
telling = alle_ongevallen.groupby(["VIAS_baannr", "jaar"]).size().rename("ongevallen").reset_index()

# Merge: traffic data on the left, so road sections without accidents are also included (these receive 0)
resultaat = verkeer.merge(telling, on=["VIAS_baannr", "jaar"], how="left")
resultaat["ongevallen"] = resultaat["ongevallen"].fillna(0)

# Vehicle-kilometres and risk
# vkm = vehicles per day x 365 days x length in km; expressed in millions
resultaat["vkm_mln"] = resultaat["intensiteit"] * 365 * resultaat["Lengte_baan_meter"] / 1000 / 1_000_000
resultaat["risico"] = resultaat["ongevallen"] / resultaat["vkm_mln"] * 100   # Accidents per 100 million vehicle-kilometres.

# UPDATED (Harmen): link the treated label using the road section number in resultaat itself.
# Previously, this used alle_ongevallen["VIAS_baannr"].map(...): pandas aligns this with resultaat by ROW INDEX,
# not by road section. As a result, only approximately 44% of rows received the correct label, and sections 1.1 to 2.1 were incorrect.
resultaat["treated"] = resultaat["VIAS_baannr"].map(snelheid_df["treated"])
resultaat

#resultaat[resultaat["VIAS_baannr"] == 3901]

In [ ]:
# 1.1
# Calculate total accidents per year for treated and comparison roads
yearly_grouped = resultaat[resultaat['treated'].isin([True, False])].groupby(
    ['jaar', 'treated']
).agg(
    total_accidents=('ongevallen', 'sum')
)

# Reshape so each treatment group becomes a separate column
accidents_by_group = yearly_grouped['total_accidents'].unstack('treated')

accidents_by_group = accidents_by_group.rename(columns={
    False: 'Comparison roads',
    True: 'Treated roads'
})

# Calculate total accidents across all road sections
yearly_total = resultaat.groupby('jaar')['ongevallen'].sum()

# Add total as a third line
accidents_by_group['Total roads'] = yearly_total

# UPDATED (Harmen): treated and comparison roads are now also shown before 2020 (these years are needed for the before-and-after
# comparison), and the COVID-19/transition years 2020–2021 are shaded in grey, as in the deliverable.

# Plot
ax = accidents_by_group.plot(kind='line', marker='o')
ax.axvspan(2019.5, 2021.5, color='0.9', zorder=0, label='COVID-19 / transition')

ax.set_xlabel('Year')
ax.set_ylabel('Total number of accidents')
ax.set_title('Total accidents: treated vs comparison vs all roads')
ax.grid(True)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 1.2
# Select the years before and after the measure
# UPDATED (Harmen): 'after' = 2022–2025 (previously 2021–2023). 2021 is still a COVID year and, according to the
# deliverables, should not be included in the main comparison; data for 2024 and 2025 is now available.
jaren_voor = [2018, 2019]
jaren_na = [2022, 2023, 2024, 2025]

# Calculate the total number of accidents per year for each group
jaarlijks = (
    resultaat[resultaat["treated"].isin([True, False])]
    .groupby(["jaar", "treated"])["ongevallen"]
    .sum()
    .unstack("treated")
    .reindex(columns=[False, True])
)

# Calculate the average annual number of accidents per group
gemiddeld_voor = jaarlijks.loc[jaren_voor].mean()
gemiddeld_na = jaarlijks.loc[jaren_na].mean()

# Create the bar chart
x = np.array([0, 1])
breedte = 0.35

fig, ax = plt.subplots(figsize=(9, 6))

ax.bar(
    x - breedte / 2,
    [gemiddeld_voor[True], gemiddeld_na[True]],
    breedte,
    label="Treated roads",
    color='C1'

)

ax.bar(
    x + breedte / 2,
    [gemiddeld_voor[False], gemiddeld_na[False]],
    breedte,
    label="Comparison roads",
    color = 'C0'
)

ax.set_xticks(x)
ax.set_xticklabels(["Before (2018–2019)", "After (2022–2025)"])
ax.set_ylabel("Average annual number of accidents")
ax.set_title("Average annual accidents before and after the 100 km/h measure")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 1.3
# Define periods
# UPDATED (Harmen): 'after' = 2022–2025 (previously 2021–2023), for the same reason as in 1.2.
jaren_voor = [2018, 2019]
jaren_na = [2022, 2023, 2024, 2025]

# Calculate total accidents per year for treated and comparison roads
jaarlijks = (
    resultaat[resultaat["treated"].isin([True, False])]
    .groupby(["jaar", "treated"])["ongevallen"]
    .sum()
    .unstack("treated")
    .reindex(columns=[False, True])
)

# Calculate average annual accidents before and after
gemiddeld_voor = jaarlijks.loc[jaren_voor].mean()
gemiddeld_na = jaarlijks.loc[jaren_na].mean()

# Create comparison table
vergelijking = pd.DataFrame({
    "Average accidents before 2020": {
        "Treated roads": gemiddeld_voor[True],
        "Comparison roads": gemiddeld_voor[False]
    },
    "Average accidents after 2020": {
        "Treated roads": gemiddeld_na[True],
        "Comparison roads": gemiddeld_na[False]
    }
})

# Calculate absolute and percentage changes
vergelijking["Absolute change"] = (
    vergelijking["Average accidents after 2020"] - vergelijking["Average accidents before 2020"]
)

vergelijking["Percentage change (%)"] = (
    vergelijking["Absolute change"]
    / vergelijking["Average accidents before 2020"] * 100
)

print(vergelijking.round(2))

In [ ]:
# 2.1
# Calculate risk per year for treated and comparison roads
yearly_grouped = resultaat[resultaat['treated'].isin([True, False])].groupby(
    ['jaar', 'treated']
).agg(
    total_accidents=('ongevallen', 'sum'),
    total_vkm_mln=('vkm_mln', 'sum')
)

yearly_grouped['overall_risk'] = (
    yearly_grouped['total_accidents'] /
    yearly_grouped['total_vkm_mln'] * 100
)

risk_by_group = yearly_grouped['overall_risk'].unstack('treated')

risk_by_group = risk_by_group.rename(columns={
    False: 'Comparison roads',
    True: 'Treated roads'
})

# Calculate total risk across all road sections
yearly_total = resultaat.groupby('jaar').agg(
    total_accidents=('ongevallen', 'sum'),
    total_vkm_mln=('vkm_mln', 'sum')
)

yearly_total['Total roads'] = (
    yearly_total['total_accidents'] /
    yearly_total['total_vkm_mln'] * 100
)

# Add total as a third line
risk_by_group['Total roads'] = yearly_total['Total roads']

# UPDATED (Harmen): treated and comparison roads are also shown before 2020, and the COVID-19/transition years are shaded in grey.

# Plot
ax = risk_by_group.plot(kind='line', marker='o')
ax.axvspan(2019.5, 2021.5, color='0.9', zorder=0, label='COVID-19 / transition')

ax.set_xlabel('Year')
ax.set_ylabel('Accidents per 100 million vehicle-km')
ax.set_title('Road accident risk: treated vs comparison vs total')
ax.grid(True)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 2.2 Traffic volume (2018–2025): traffic intensity for treated and comparison roads
# For each group, calculate the length-weighted average number of vehicles per day (= total vehicle-kilometres / total length),
# so that a long road section carries more weight than a short one. On the right, show the values as an index (2019 = 100),
# so the changes in both groups can be compared directly: this illustrates why adjusting for traffic volume is necessary.
v = resultaat[resultaat["treated"].isin([True, False])].copy()
v["int_x_lengte"] = v["intensiteit"] * v["Lengte_baan_meter"]
som = v.groupby(["jaar", "treated"])[["int_x_lengte", "Lengte_baan_meter"]].sum()
intensiteit_groep = (som["int_x_lengte"] / som["Lengte_baan_meter"]).unstack("treated")
intensiteit_groep = intensiteit_groep.rename(columns={False: "Comparison roads", True: "Treated roads"})

index_2019 = intensiteit_groep / intensiteit_groep.loc[2019] * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
intensiteit_groep.plot(ax=axes[0], marker="o")
axes[0].set_ylabel("Average vehicles per day (per direction)")
axes[0].set_title("Traffic volume on treated and comparison roads")
index_2019.plot(ax=axes[1], marker="o")
axes[1].axhline(100, color="0.5", linewidth=1)
axes[1].set_ylabel("Index (2019 = 100)")
axes[1].set_title("Traffic volume relative to 2019")
for ax in axes:
    ax.axvspan(2019.5, 2021.5, color="0.9", zorder=0, label="COVID-19 / transition")
    ax.set_xlabel("Year")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

index_2019.round(1)

In [ ]:
# Helper function for 2.3 and 2.4: Poisson regression with fixed effects for each road section and year
#
# Model:  log E[accidents] = log(vehicle-km) + road_section_effect + year_effect + beta * term
#   - log(vehicle-km) as an 'offset': we model the RISK (accidents per kilometre driven), not the number of accidents
#   - road-section effect: fixed differences between road sections (e.g. busy urban ring roads) are accounted for
#   - year effect: anything that affected ALL motorways (COVID-19, reporting, weather) is accounted for
#   - exp(beta) = incidence-rate ratio (IRR): the change in risk on treated roads divided by the change in risk
#     on comparison roads. IRR 0.80 = the risk on treated roads decreased 20% more than on comparison roads.
#
# With approximately 2,600 road sections, a standard regression with a dummy variable for each road section is slow
# (approximately 1 minute per model). ConditionalPoisson (statsmodels) estimates exactly the same model, but
# conditions out the road-section effects, completing in less than a second.
# The standard errors are cluster-robust at the road-section level:
# accidents on the same road section are correlated over the years, and accident counts often vary more than
# a Poisson distribution assumes (overdispersion). This makes Poisson regression with robust errors suitable here
# (instead of negative binomial regression).
# Check: a standard GLM with a dummy variable for each road section produced the same IRR (0.794) and almost the same interval.

def poisson_fe(data, termen):
    """Estimate the model for the columns in `termen`; return the IRR, 95% CI, and p-value for each term."""
    # Road sections where the number of accidents never changes across the selected years (e.g. zero every year)
    # provide no information about a change; ConditionalPoisson also excludes them. We remove them beforehand.
    data = data[data.groupby("VIAS_baannr")["ongevallen"].transform("nunique") > 1]
    jaar_dummies = pd.get_dummies(data["jaar"], prefix="jaar", drop_first=True, dtype=float)
    X = pd.concat([data[termen].astype(float), jaar_dummies], axis=1).to_numpy()
    y = data["ongevallen"].to_numpy(float)
    offset = np.log(data["vkm_mln"].to_numpy(float))
    baanvak = data["VIAS_baannr"].to_numpy()

    fit = ConditionalPoisson(y, X, groups=baanvak, offset=offset).fit(method="bfgs", maxiter=1000)
    b = fit.params

    # Cluster-robust standard errors at the road-section level (sandwich formula)
    eta = X @ b + offset
    e = np.exp(eta - pd.Series(eta).groupby(baanvak).transform("max").to_numpy())
    p = e / pd.Series(e).groupby(baanvak).transform("sum").to_numpy()   # expected share of each year within a road section
    n = pd.Series(y).groupby(baanvak).transform("sum").to_numpy()        # total number of accidents per road section
    x_c = X - pd.DataFrame(X * p[:, None]).groupby(baanvak).transform("sum").to_numpy()
    scores = pd.DataFrame(x_c * (y - n * p)[:, None]).groupby(baanvak).sum().to_numpy()
    H = -(x_c * (n * p)[:, None]).T @ x_c
    G = len(np.unique(baanvak))
    H_inv = np.linalg.inv(H)
    se = np.sqrt(np.diag(H_inv @ (scores.T @ scores) @ H_inv * G / (G - 1)))

    k = len(termen)
    return pd.DataFrame({
        "term": termen,
        "IRR": np.exp(b[:k]),
        "CI_low": np.exp(b[:k] - 1.96 * se[:k]),
        "CI_high": np.exp(b[:k] + 1.96 * se[:k]),
        "p_value": 2 * norm.sf(np.abs(b[:k] / se[:k])),
    })

# Model data: only treated and comparison road sections with traffic (vkm > 0),
# excluding road sections with no accidents in any year (as they provide no information about a change)
model_data = resultaat[resultaat["treated"].isin([True, False]) & (resultaat["vkm_mln"] > 0)].copy()
model_data = model_data[model_data.groupby("VIAS_baannr")["ongevallen"].transform("sum") > 0]
model_data["behandeld"] = (model_data["treated"] == True).astype(int)
print(f"modeldata: {model_data['VIAS_baannr'].nunique()} road sections, {len(model_data)} road section-years")

In [ ]:
# 2.3 Event study (2018–2025): difference between treated and comparison roads by year, using 2019 as the reference
# For each year except 2019, create a 'treated x year' term. The IRR for a year is:
#     (risk on treated roads / risk on comparison roads in that year) /
#     (the same ratio in 2019)
#   - Before the intervention (2018), the IRR should be around 1: this means both groups followed similar trends
#     (this is the key assumption of difference-in-differences)
#   - After the intervention, the IRR shows how much the risk on treated roads changed additionally
REFERENTIEJAAR = 2019
jaren = sorted(j for j in model_data["jaar"].unique() if j != REFERENTIEJAAR)
for j in jaren:
    model_data[f"behandeld_{j}"] = model_data["behandeld"] * (model_data["jaar"] == j)

event_study = poisson_fe(model_data, [f"behandeld_{j}" for j in jaren])
event_study["jaar"] = jaren
referentie = pd.DataFrame({"term": ["referentie"], "IRR": [1.0], "CI_low": [1.0], "CI_high": [1.0],
                           "p_value": [np.nan], "jaar": [REFERENTIEJAAR]})
event_study = pd.concat([event_study, referentie]).sort_values("jaar").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.axhline(1, color="0.5", linewidth=1)
ax.errorbar(event_study["jaar"], event_study["IRR"],
            yerr=[event_study["IRR"] - event_study["CI_low"], event_study["CI_high"] - event_study["IRR"]],
            fmt="o", capsize=4, color="C1", label="IRR treated vs comparison (95% CI)")
ax.axvspan(2019.5, 2021.5, color="0.9", zorder=0, label="COVID-19 / transition")
ax.set_xlabel("Year")
ax.set_ylabel("Incidence-rate ratio (2019 = 1)")
ax.set_title("Accident risk on treated roads relative to comparison roads")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

event_study[["jaar", "IRR", "CI_low", "CI_high", "p_value"]].round(3)

In [ ]:
# 2.4 Difference-in-Differences (DiD): one number for the relative change in accident risk
# Before = 2018–2019, after = 2022–2025; the COVID-19/transition years 2020–2021 are excluded.
# treated_after = 1 for treated roads in the years after the intervention, and 0 otherwise.
# Model: Poisson regression with vehicle-km as exposure, and road-section and year effects (see the helper function above).
def did(data, voor, na):
    d = data[data["jaar"].isin(voor + na)].copy()
    d["behandeld_na"] = d["behandeld"] * d["jaar"].isin(na)
    return poisson_fe(d, ["behandeld_na"]).iloc[0]

VOOR = [2018, 2019]
NA = [2022, 2023, 2024, 2025]
did_resultaten = {"Main: before 2018-2019, after 2022-2025": did(model_data, VOOR, NA)}

# Sensitivity analysis 1: the number of REGISTERED accidents increased sharply from 2018 to 2019,
# while traffic volume barely changed (reporting effect). Therefore, also use 2019 alone as the 'before' period.
did_resultaten["Before = 2019 only"] = did(model_data, [2019], NA)

# Sensitivity analysis 2: a stricter comparison group. A median of 100 allows nearly half of the
# accidents before 2020 to have occurred under a 120/130 km/h speed limit. Here, include only comparison roads
# with no 120/130 km/h speed limits, at least 80% of observations at 100 km/h, and at least 5 observations
# with a recorded speed limit in 2018–2019.
voor_2020 = alle_ongevallen[alle_ongevallen["jaar"].isin([2018, 2019])].dropna(subset=["limiet"])
per_baanvak = voor_2020.groupby("VIAS_baannr")["limiet"].agg(
    n="size", hoog=lambda s: s.isin([120, 130]).mean(), honderd=lambda s: (s == 100).mean())
strenge_comparison = per_baanvak[(per_baanvak["hoog"] == 0) & (per_baanvak["honderd"] >= 0.8)
                                 & (per_baanvak["n"] >= 5)].index
streng = model_data[(model_data["behandeld"] == 1) | model_data["VIAS_baannr"].isin(strenge_comparison)]
did_resultaten["Stricter comparison group"] = did(streng, VOOR, NA)

# Table: IRR, 95% confidence interval, p-value, and percentage change
did_tabel = pd.DataFrame(did_resultaten).T
did_tabel = pd.DataFrame({
    "IRR": did_tabel["IRR"].astype(float).round(3),
    "95% CI": [f"{lo:.3f} – {hi:.3f}" for lo, hi in zip(did_tabel["CI_low"], did_tabel["CI_high"])],
    "p-value": did_tabel["p_value"].astype(float).map(lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"),
    "Change in risk vs comparison": (did_tabel["IRR"].astype(float) - 1).map("{:+.1%}".format),
})
print(f"comparison roads: {model_data.loc[model_data['behandeld'] == 0, 'VIAS_baannr'].nunique()} (main), "
      f"{streng.loc[streng['behandeld'] == 0, 'VIAS_baannr'].nunique()} (stricter)")
did_tabel